In [8]:
using Random
using DynamicPolynomials
using SumOfSquares
import CSDP
using JuMP, MosekTools, LinearAlgebra
using Statistics

### Random Polynomial Generation

In [2]:
"""
random_sos_poly(varlists; degree=8, nterms=16, seed=33)

Generate a random polynomial suitable as an SOS test instance.
- `varlists`: vector of variable arrays (e.g. `[x,y]` where `x` and `y` are @polyvar arrays).
- `degree`: maximum total degree of each term.
- `nterms`: number of distinct monomial terms to include.
- Coefficients are sampled uniformly from the continuous interval [-1, 1].
- `seed`: RNG seed for reproducibility.

Returns a DynamicPolynomials polynomial built from the provided variables.
"""
function random_sos_poly(varlists::AbstractVector{<:AbstractVector}; degree::Int=8, nterms::Int=16, seed::Integer=33)
    Random.seed!(seed)
    allvars = reduce(vcat, varlists)
    n = length(allvars)
    poly = zero(allvars[1])

    exps_set = Set{Tuple{Vararg{Int}}}()
    while length(exps_set) < nterms
        d = rand(1:degree)
        exps = zeros(Int, n)
        for _ in 1:d
            idx = rand(1:n)
            exps[idx] += 1
        end
        # only accept if total degree within limit and not the zero vector
        if 1 <= sum(exps) <= degree
            push!(exps_set, tuple(exps...))
        end
    end

    for tup in exps_set
        coeff = round(rand()*2.0 - 1.0, digits=2)
        # build monomial (skip variables with exponent 0)
        term = coeff * prod(allvars[i]^tup[i] for i in 1:n if tup[i] > 0)
        poly += term
    end

    # include a random constant in [-1,1]
    poly += round(rand()*2.0 - 1.0, digits=2)

    return poly
end

random_sos_poly

In [3]:
"""
random_sos_poly_reduced(varlists; degree=8, nterms=16, seed=33)

Generate a random polynomial using full variables, then substitute out the last 
variable of each group using the constraint sum(vars) = 1.
Returns (reduced_poly, reduced_varlists) where reduced_varlists uses first n-1 vars of each group.
"""
function random_sos_poly_reduced(varlists::AbstractVector{<:AbstractVector}; degree::Int=8, nterms::Int=16, seed::Integer=33)
    # First generate the polynomial with all variables
    p = random_sos_poly(varlists; degree=degree, nterms=nterms, seed=seed)
    
    # Reduced varlists: just use first n-1 variables of each group
    reduced_varlists = [group[1:end-1] for group in varlists]
    
    # Substitute: last var -> 1 - sum(first n-1 vars)
    for group in varlists
        last_var = group[end]
        substitution = 1 - sum(group[1:end-1])
        p = subs(p, last_var => substitution)
    end
    
    return p, reduced_varlists
end

"""
build_reduced_constraints(varlists_reduced)

Build the constraint set for reduced variables where each group satisfies:
- x_i >= 0 for i = 1..n-1
- x_n = 1 - sum(x[1:n-1]) >= 0  (i.e., sum <= 1)
- 1 - sum(x_i^2) - x_n^2 >= 0  (norm constraint)
"""
function build_reduced_constraints(varlists_reduced::AbstractVector)
    # Get a sample variable to determine polynomial type
    allvars = reduce(vcat, varlists_reduced)
    zp = zero(allvars[1])  # zero polynomial for type conversion
    op = one(allvars[1])   # one polynomial
    
    gs = typeof(zp)[]
    
    for vars in varlists_reduced
        # vars[i] >= 0 for each reduced variable
        for v in vars
            push!(gs, v + zp)
        end
        # last var = 1 - sum(vars) >= 0
        push!(gs, op - sum(vars))
        # norm constraint: 1 - sum(vars.^2) - (1-sum(vars))^2 >= 0
        last_var_expr = op - sum(vars)
        push!(gs, op - sum(vars.^2) - last_var_expr^2)
    end
    
    return basic_semialgebraic_set(FullSpace(), gs)
end

build_reduced_constraints

### SOS Solver

In [4]:
# Solve polynomial p using REDUCED variables (fewer vars, no equality constraints)
# Returns: (nu_meas, tval, status, t_reduce, t_solver, t_total)
#   t_reduce: time for variable substitution + constraint setup
#   t_solver: Mosek solve_time
#   t_total:  wall time including extraction
function solve_polynomial_sos_reduced(p, varlists; d_sel, tol)
    t0 = time()

    # Substitute last var -> 1 - sum(first n-1 vars)
    p_sub = p
    reduced_varlists = [group[1:end-1] for group in varlists]
    for group in varlists
        p_sub = subs(p_sub, group[end] => 1 - sum(group[1:end-1]))
    end
    reduced_varlists = [group[1:end-1] for group in varlists]
    
    Sg_reduced = build_reduced_constraints(reduced_varlists)

    t_reduce = time() - t0

    model = SOSModel(Mosek.Optimizer)
    set_optimizer_attribute(model, "MSK_IPAR_LOG", 0)
    @variable(model, t)
    @objective(model, Min, t)
    @constraint(model, c, p_sub <= t, domain = Sg_reduced, maxdegree = d_sel)

    optimize!(model)
    t_solver = solve_time(model)
    status = termination_status(model)

    tval = value(t)

    nu = moment_matrix(model[:c])
    nu_meas = atomic_measure(nu, tol) 

    t_total = time() - t0
      
    println("Atomic measure: $nu_meas")
    println("Status: $status, Value: $(round(tval, digits=6)) [REDUCED] | reduce=$(round(t_reduce, digits=4))s, solver=$(round(t_solver, digits=4))s, total=$(round(t_total, digits=4))s")

    return nu_meas, tval, status, t_reduce, t_solver, t_total

end

solve_polynomial_sos_reduced (generic function with 1 method)

In [5]:
# Test SOS performance on reduced polynomials
function test_random_polynomials_reduced(n_tests, varlists; degree, nterms, d_sel, tol, seed)
    sos_success = 0
    sos_fail = 0
    failing_tests = []
    records = []
    all_reduce = Float64[]
    all_solver = Float64[]
    all_total = Float64[]
    for i in 1:n_tests
        s = seed + i - 1
        p = random_sos_poly(varlists; degree=degree, nterms=nterms, seed=s)
        println('-'^60)
        println("Test $i — seed=$s [REDUCED]")
        println("Polynomial: $p")
        nu, tval, status, t_reduce, t_solver, t_total = solve_polynomial_sos_reduced(p, varlists; d_sel=d_sel, tol=tol)
        push!(all_reduce, t_reduce)
        push!(all_solver, t_solver)
        push!(all_total, t_total)
        has_atoms = (nu !== nothing)
        if has_atoms
            println("  SOS extraction succeeded; val = $(round(tval,digits=6)); total=$(round(t_total,digits=4))s")
            sos_success += 1
        else
            println("  SOS extraction failed or no atoms; bound val = $(round(tval,digits=6)); total=$(round(t_total,digits=4))s")
            sos_fail += 1
            push!(failing_tests, (i=i, seed=s, tval=tval, status=status))
        end
        push!(records, (i, s, tval, has_atoms, t_reduce, t_solver, t_total, status))
    end
    println()
    println('='^60)
    println("SUMMARY [REDUCED]: SOS found atoms: $sos_success / $n_tests ; failed: $sos_fail / $n_tests")
    println("Reduce time: total=$(round(sum(all_reduce), digits=2))s, avg=$(round(mean(all_reduce), digits=4))s")
    println("Solver time: total=$(round(sum(all_solver), digits=2))s, avg=$(round(mean(all_solver), digits=4))s")
    println("Total time:  total=$(round(sum(all_total), digits=2))s, avg=$(round(mean(all_total), digits=4))s")
    if !isempty(failing_tests)
        println("Failing tests:")
        for fail in failing_tests
            println("  Test $(fail.i) (seed $(fail.seed)): bound = $(round(fail.tval, digits=6)), status = $(fail.status)")

        end
    end
    
    return records

end    

test_random_polynomials_reduced (generic function with 1 method)

### Parameter Setting

In [6]:
@polyvar x[1:3] y[1:3] z[1:3] w[1:3] u[1:3] v[1:3] var7[1:3] var8[1:3] var9[1:3] var10[1:3]
varlists = [x,y]
varlists_var3 = [x,y,z]
varlists_var4 = [x,y,z,w]
varlists_var5 = [x,y,z,w,u]
varlists_var6 = [x,y,z,w,u,v]
varlists_var7 = [x,y,z,w,u,v,var7]
varlists_var8 = [x,y,z,w,u,v,var7,var8]
varlists_var9 = [x,y,z,w,u,v,var7,var8,var9]
varlists_var10 = [x,y,z,w,u,v,var7,var8,var9,var10]
n_tests = 50
seed = 33

33

### Single-player IREFGs

In [73]:
records = test_random_polynomials_reduced(n_tests, varlists; degree=4, nterms=16, d_sel=4, tol=1e-1, seed)

------------------------------------------------------------
Test 1 — seed=33 [REDUCED]
Polynomial: -0.91 - 0.5*y[3] + 0.17*x[1] + 0.75*y[2]*y[3] - 0.52*y[1]*y[3] + 0.68*x[3]*y[1] + 0.45*x[3]*y[1]*y[3] + 0.67*x[3]^2*y[3] + 0.77*x[2]*y[2]*y[3] + 0.52*x[2]*y[1]*y[3] + 0.75*x[2]*y[1]*y[2] - 0.08*x[1]*y[1]*y[3] - 0.95*x[1]^2*y[1] - 0.66*y[2]^3*y[3] - 0.62*x[3]*y[2]^3 + 0.67*x[2]*y[3]^3 - 0.53*x[1]*x[2]^2*x[3]
Atomic measure: Atomic measure on the variables x[1], x[2], y[1], y[2] with 1 atoms:
 at [0.0, 2.080848663273264e-8, 0.9999999802910022, 0.0] with weight 0.9999999882999062
Status: OPTIMAL, Value: -0.23 [REDUCED] | reduce=0.0s, solver=0.0078s, total=0.034s
  SOS extraction succeeded; val = -0.23; total=0.034s
------------------------------------------------------------
Test 2 — seed=34 [REDUCED]
Polynomial: 0.56 + 0.62*y[2] + 0.57*y[1] - 0.32*y[1]*y[2] + 0.74*x[3]*y[2] - 0.39*x[1]*y[3] - 0.8*x[1]*y[2] + 0.4*x[1]*y[1] - 0.92*y[1]*y[2]^2 + 0.69*x[2]*x[3]*y[2] + 0.97*x[1]*y[2]*y[3] - 0.6

50-element Vector{Any}:
 (1, 33, -0.23000000490551617, true, 0.0, 0.0077571, 0.03399991989135742, OPTIMAL)
 (2, 34, 1.919999998950227, true, 0.0, 0.0079344, 0.009999990463256836, OPTIMAL)
 (3, 35, 1.2799999995969118, true, 0.0, 0.0074719, 0.024000167846679688, OPTIMAL)
 (4, 36, 1.3499999974514056, true, 0.0, 0.0077738, 0.01699995994567871, OPTIMAL)
 (5, 37, 2.2200000003893536, true, 0.0, 0.0068985, 0.009999990463256836, OPTIMAL)
 (6, 38, 1.66999999466565, true, 0.0, 0.0077181, 0.012000083923339844, OPTIMAL)
 (7, 39, 0.6574999967030911, true, 0.0, 0.0083171, 0.03099989891052246, OPTIMAL)
 (8, 40, 0.4199999975846454, true, 0.0, 0.007396, 0.018000125885009766, OPTIMAL)
 (9, 41, 1.917611255870268e-9, true, 0.0, 0.0072598, 0.016000032424926758, OPTIMAL)
 (10, 42, 0.21999999184309035, true, 0.0, 0.0073629, 0.01699995994567871, OPTIMAL)
 ⋮
 (42, 74, -0.18000000194177532, true, 0.0, 0.0078888, 0.019999980926513672, OPTIMAL)
 (43, 75, 1.6799999858414087, true, 0.0, 0.0077332, 0.0169999599456787

In [74]:
records = test_random_polynomials_reduced(n_tests, varlists_var3; degree=4, nterms=16, d_sel=4, tol=3e-1, seed)

------------------------------------------------------------
Test 1 — seed=33 [REDUCED]
Polynomial: -0.66 - 0.52*z[3] - 0.5*x[2] - 0.08*x[1] + 0.17*z[2]*z[3] - 0.17*y[3]*z[3] + 0.49*y[2]^2 + 0.68*y[2]*y[3]*z[3] + 0.67*y[1]*y[2]*z[3] - 0.53*x[3]*y[2]*z[1] + 0.67*x[2]*z[1]*z[3] - 0.95*x[2]*y[3]*z[3] + 0.75*x[2]^2*y[2] + 0.9*x[1]*y[3]*z[3] + 0.52*z[1]*z[2]^2*z[3] + 0.37*x[3]*z[2]*z[3]^2 + 0.59*x[1]*x[2]*x[3]*y[1]
Atomic measure: Atomic measure on the variables x[1], x[2], y[1], y[2], z[1], z[2] with 1 atoms:
 at [4.795309071881855e-8, 0.9999999248383611, 1.5520692002892338e-8, 0.9999999711665568, 0.8877924465104511, 0.00029081617540948485] with weight 0.9999997960032567
Status: OPTIMAL, Value: 0.088395 [REDUCED] | reduce=0.0s, solver=0.0283s, total=0.071s
  SOS extraction succeeded; val = 0.088395; total=0.071s
------------------------------------------------------------
Test 2 — seed=34 [REDUCED]
Polynomial: 0.97 + 0.76*z[1] - 0.19*y[3] + 0.64*y[2]*z[1] - 0.92*y[1]*z[1] - 0.6*x[2]*z[2] -

50-element Vector{Any}:
 (1, 33, 0.08839542444306787, true, 0.0, 0.0283119, 0.0709998607635498, OPTIMAL)
 (2, 34, 2.939999998764393, true, 0.0, 0.0161746, 0.046000003814697266, OPTIMAL)
 (3, 35, 0.7299999969849887, true, 0.0009999275207519531, 0.0163548, 0.05799984931945801, OPTIMAL)
 (4, 36, 1.2599999999636602, true, 0.0009999275207519531, 0.0171086, 0.039999961853027344, OPTIMAL)
 (5, 37, 0.009999998930073839, true, 0.0, 0.0140369, 0.03900003433227539, OPTIMAL)
 (6, 38, 1.7299999524868346, true, 0.0, 0.0186322, 0.05800008773803711, OPTIMAL)
 (7, 39, 1.3700724696126296, true, 0.0009999275207519531, 0.0269824, 0.05299997329711914, OPTIMAL)
 (8, 40, -0.17000003296209681, true, 0.0010001659393310547, 0.016697, 0.04900002479553223, OPTIMAL)
 (9, 41, 0.07548443351222545, true, 0.0010001659393310547, 0.0205412, 0.04800009727478027, OPTIMAL)
 (10, 42, 0.6500021980052187, true, 0.0, 0.0287381, 0.06100010871887207, OPTIMAL)
 ⋮
 (42, 74, 1.2012903204646077, true, 0.0, 0.0179056, 0.0639998912811

In [14]:
records = test_random_polynomials_reduced(n_tests, varlists_var4; degree=4, nterms=16, d_sel=4, tol=3e-1, seed)

------------------------------------------------------------
Test 1 — seed=33 [REDUCED]
Polynomial: 0.67 + 0.05*w[3] + 0.59*x[2] + 0.49*x[1] + 0.9*w[1]*w[3] + 0.52*z[2]*w[3] - 0.17*z[1]*w[2] + 0.67*y[3]*z[2]*w[2] - 0.52*y[3]^2*w[3] + 0.37*y[1]*z[1]*z[3] + 0.68*x[3]*z[3]*w[2] - 0.03*x[3]*z[2]*w[2] - 0.5*x[2]^2*z[1] - 0.08*x[1]*z[2]*w[3] - 0.95*z[3]*w[1]^2*w[3] - 0.1*y[1]*w[2]^2*w[3] + 0.75*x[1]*x[3]*y[1]*y[3]
Atomic measure: Atomic measure on the variables x[1], x[2], y[1], y[2], z[1], z[2], w[1], w[2] with 1 atoms:
 at [0.0, 0.9999999994966631, 0.0, 0.0, 0.0, 0.999999999820754, 0.0, 0.9999999987248075] with weight 0.9999999995656726
Status: OPTIMAL, Value: 1.93 [REDUCED] | reduce=0.001s, solver=0.0572s, total=0.397s
  SOS extraction succeeded; val = 1.93; total=0.397s
------------------------------------------------------------
Test 2 — seed=34 [REDUCED]
Polynomial: 0.4 - 0.19*w[1] - 0.8*z[2] - 0.75*z[1] + 0.69*z[1]*z[3] + 0.76*y[2]*z[3] + 0.45*x[2]*w[2] - 0.32*x[1]*z[1] - 0.92*z[1]*z[

50-element Vector{Any}:
 (1, 33, 1.9299999993935688, true, 0.0009999275207519531, 0.0571705, 0.3970000743865967, OPTIMAL)
 (2, 34, 1.609999997823166, true, 0.0009999275207519531, 0.0569924, 0.18400001525878906, OPTIMAL)
 (3, 35, 2.6799999999786355, true, 0.0, 0.0583865, 0.20199990272521973, OPTIMAL)
 (4, 36, 1.8299999804664075, true, 0.0009999275207519531, 0.0988768, 0.23199987411499023, OPTIMAL)
 (5, 37, 0.28999998735011057, true, 0.0, 0.0625661, 0.18400001525878906, OPTIMAL)
 (6, 38, 2.91369790976191, true, 0.0, 0.0745073, 0.21399998664855957, OPTIMAL)
 (7, 39, 2.489999999987141, true, 0.0, 0.0585771, 0.18400001525878906, OPTIMAL)
 (8, 40, 0.14999999966514607, true, 0.0, 0.0573758, 0.20000004768371582, OPTIMAL)
 (9, 41, -0.6884097354270706, true, 0.0, 0.0897687, 0.21700000762939453, OPTIMAL)
 (10, 42, 1.0104994925321658, true, 0.0, 0.1156578, 0.247999906539917, OPTIMAL)
 ⋮
 (42, 74, 1.195208162431369, true, 0.0, 0.0713404, 0.21700000762939453, OPTIMAL)
 (43, 75, 1.5799999972820233, t

In [14]:
records = test_random_polynomials_reduced(n_tests, varlists_var5; degree=4, nterms=16, d_sel=4, tol=3e-1, seed)

------------------------------------------------------------
Test 1 — seed=33 [REDUCED]
Polynomial: 0.67 + 0.49*u[3] - 0.5*x[3] - 0.03*x[2] + 0.52*u[1]*u[3] + 0.68*w[1]*u[2] + 0.75*z[3]*u[2] - 0.1*z[2]*w[1]*u[2] - 0.08*z[1]^2*u[2] + 0.9*y[2]*z[3]*w[2] - 0.17*y[1]*w[2]*u[2] - 0.95*x[3]*w[1]*u[2] + 0.05*x[2]*x[3]*z[2] + 0.59*x[1]*w[1]*u[3] + 0.37*w[2]*w[3]*u[1]*u[3] + 0.67*y[2]*u[1]*u[2]*u[3] - 0.52*x[2]*y[1]*y[2]*z[1]
Atomic measure: Atomic measure on the variables x[1], x[2], y[1], y[2], z[1], z[2], w[1], w[2], u[1], u[2] with 1 atoms:
 at [0.9999998111011752, 1.847125735440223e-7, 0.0, 0.9999999833611416, 0.0, 0.0, 3.525271782925123e-8, 0.9999999630367006, 0.0, 0.999999979550158] with weight 0.9999999738365809
Status: OPTIMAL, Value: 2.32 [REDUCED] | reduce=0.001s, solver=0.1933s, total=0.839s
  SOS extraction succeeded; val = 2.32; total=0.839s
------------------------------------------------------------
Test 2 — seed=34 [REDUCED]
Polynomial: 0.4 + 0.62*w[3] + 0.45*w[1] + 0.32*z[3] -

50-element Vector{Any}:
 (1, 33, 2.319999949377871, true, 0.0009999275207519531, 0.1932709, 0.8389999866485596, OPTIMAL)
 (2, 34, 2.350056939933417, true, 0.002000093460083008, 0.3883813, 1.00600004196167, OPTIMAL)
 (3, 35, 1.375226522791444, true, 0.002000093460083008, 0.3693924, 0.9900000095367432, OPTIMAL)
 (4, 36, 2.939999997646463, true, 0.0010001659393310547, 0.175007, 0.7970001697540283, OPTIMAL)
 (5, 37, 1.6199999992838996, true, 0.0009999275207519531, 0.1580227, 0.7730000019073486, OPTIMAL)
 (6, 38, 2.6999999261995553, true, 0.0009999275207519531, 0.2046647, 0.8250000476837158, OPTIMAL)
 (7, 39, 1.2999999999995333, true, 0.0009999275207519531, 0.1873975, 0.8050000667572021, OPTIMAL)
 (8, 40, 1.860020084364372, true, 0.0019998550415039062, 0.3482894, 0.9700000286102295, OPTIMAL)
 (9, 41, 1.5174999998125087, true, 0.0019998550415039062, 0.1654766, 0.7809998989105225, OPTIMAL)
 (10, 42, 1.080069363194212, true, 0.0009999275207519531, 0.3487957, 0.9659998416900635, OPTIMAL)
 ⋮
 (4

In [15]:
records = test_random_polynomials_reduced(n_tests, varlists_var6; degree=4, nterms=16, d_sel=4, tol=3e-1, seed)

------------------------------------------------------------
Test 1 — seed=33 [REDUCED]
Polynomial: 0.67 - 0.08*v[2] + 0.37*x[3] + 0.68*x[2] + 0.05*u[3]*v[3] + 0.75*w[3]*v[2] + 0.49*w[2]*v[2] + 0.59*z[3]*w[3]*v[2] - 0.1*z[2]*z[3]*v[2] + 0.67*y[3]*w[1]*u[1] - 0.03*y[1]*u[1]*v[2] + 0.9*y[1]*w[2]*v[2] - 0.5*x[3]^2*w[1] - 0.95*x[1]*w[3]*v[2] - 0.17*u[1]*u[3]^2*v[3] + 0.52*y[3]*v[1]*v[2]*v[3] - 0.52*x[2]*y[1]*y[3]*z[2]
Atomic measure: Atomic measure on the variables x[1], x[2], y[1], y[2], z[1], z[2], w[1], w[2], u[1], u[2], v[1], v[2] with 1 atoms:
 at [2.9278729646711445e-8, 0.9999999199747173, 0.9999996215603923, 1.2530565225841014e-7, 0.5252668191451523, 0.16169572131321677, 8.284043679924472e-8, 0.9975972629542795, 4.870968610684149e-7, 0.3219272411305355, 3.559867303812967e-8, 0.9999999185755644] with weight 1.0010608715532243
Status: OPTIMAL, Value: 2.660589 [REDUCED] | reduce=0.002s, solver=1.0563s, total=3.942s
  SOS extraction succeeded; val = 2.660589; total=3.942s
--------------

50-element Vector{Any}:
 (1, 33, 2.660588773481341, true, 0.002000093460083008, 1.0563056, 3.941999912261963, OPTIMAL)
 (2, 34, 1.550029909172157, true, 0.002000093460083008, 1.0251156, 3.88700008392334, OPTIMAL)
 (3, 35, 2.5499999999970244, true, 0.002000093460083008, 0.4787772, 3.3450000286102295, OPTIMAL)
 (4, 36, 3.289999999988053, true, 0.002000093460083008, 0.5046038, 3.38100004196167, OPTIMAL)
 (5, 37, 1.1699998250228674, true, 0.002000093460083008, 0.8432806, 3.748000144958496, OPTIMAL)
 (6, 38, 3.7099999999991273, true, 0.0009999275207519531, 0.490259, 3.384000062942505, OPTIMAL)
 (7, 39, 1.6689360132490862, true, 0.003000020980834961, 1.0551926, 3.94599986076355, OPTIMAL)
 (8, 40, 2.1003474217318954, true, 0.002000093460083008, 0.9606602, 3.865000009536743, OPTIMAL)
 (9, 41, 1.2444004089398666, true, 0.002000093460083008, 0.9599291, 3.865000009536743, OPTIMAL)
 (10, 42, 1.930014559895616, true, 0.002000093460083008, 0.9935089, 3.9110000133514404, OPTIMAL)
 ⋮
 (42, 74, 0.93106

In [18]:
records = test_random_polynomials_reduced(n_tests, varlists; degree=6, nterms=16, d_sel=6, tol=1e-1, seed)

------------------------------------------------------------
Test 1 — seed=33 [REDUCED]
Polynomial: 0.12 + 0.21*y[1] + 0.49*x[1] - 0.01*y[3]^2 + 0.12*y[2]*y[3]^2 + 0.18*y[1]^2*y[3] - 0.9*x[3]*y[2]^2*y[3] + 0.33*x[2]*x[3]*y[3]^2 + 0.77*x[2]^2*y[2]*y[3] + 0.89*x[1]*y[1]^2*y[3] + 0.44*x[2]^2*y[1]*y[2]*y[3] + 0.7*x[1]*x[2]*x[3]*y[2]*y[3] + 0.88*x[3]^2*y[1]^3*y[3] - 0.17*x[2]^3*y[1]*y[3]^2 - 0.22*x[1]*x[2]*y[2]^3*y[3] - 0.94*x[1]*x[2]^2*x[3]^3 - 0.06*x[1]^2*x[2]*y[1]*y[2]*y[3]
Atomic measure: Atomic measure on the variables x[1], x[2], y[1], y[2] with 1 atoms:
 at [0.9999999871631894, 0.0, 0.7552930311344725, 1.5167666379284277e-8] with weight 1.0000000015703612
Status: OPTIMAL, Value: 0.917382 [REDUCED] | reduce=0.001s, solver=0.0349s, total=0.083s
  SOS extraction succeeded; val = 0.917382; total=0.083s
------------------------------------------------------------
Test 2 — seed=34 [REDUCED]
Polynomial: -0.59 - 0.6*y[3] + 0.4*y[2] + 0.6*x[2] - 0.66*x[3]*y[2] + 0.56*x[2]*y[1] + 0.32*y[1]^2*y

50-element Vector{Any}:
 (1, 33, 0.9173818991206004, true, 0.0010001659393310547, 0.0349207, 0.08299994468688965, OPTIMAL)
 (2, 34, 0.5699999987350527, true, 0.0, 0.031161, 0.0839998722076416, OPTIMAL)
 (3, 35, -0.012500004011703918, true, 0.0, 0.0386056, 0.09599995613098145, OPTIMAL)
 (4, 36, 1.5400000000651053, true, 0.0, 0.0278949, 0.06999993324279785, OPTIMAL)
 (5, 37, 0.23413786671336673, true, 0.0, 0.0307138, 0.0989999771118164, OPTIMAL)
 (6, 38, 1.0199999932745103, true, 0.0, 0.0473647, 0.08299994468688965, OPTIMAL)
 (7, 39, 0.285953292140397, true, 0.0, 0.0353001, 0.10199999809265137, OPTIMAL)
 (8, 40, 1.509999900292471, true, 0.0009999275207519531, 0.0308723, 0.08500003814697266, OPTIMAL)
 (9, 41, 0.20999998709485473, true, 0.0009999275207519531, 0.0307146, 0.07999992370605469, OPTIMAL)
 (10, 42, 1.079999975073478, true, 0.0, 0.0304229, 0.08299994468688965, OPTIMAL)
 ⋮
 (42, 74, 0.6999999987644506, true, 0.0, 0.0269618, 0.08500003814697266, OPTIMAL)
 (43, 75, 1.001957812585487

In [23]:
records = test_random_polynomials_reduced(n_tests, varlists; degree=8, nterms=16, d_sel=8, tol=3e-1, seed)

------------------------------------------------------------
Test 1 — seed=33 [REDUCED]
Polynomial: 0.96 - 0.77*x[3]*y[3] + 0.33*x[1]*y[2] + 0.84*x[2]*y[2]*y[3]^2 - 0.16*x[1]*x[3]*y[1]*y[2]*y[3] + 0.31*x[3]*y[1]^3*y[3]^2 - 0.31*x[2]^2*y[1]*y[2]*y[3]^2 - 0.29*x[1]*y[1]^3*y[2]*y[3] - 0.34*x[1]*x[2]*y[1]*y[2]*y[3]^2 - 0.26*x[1]*x[2]^2*x[3]*y[1]*y[3] - 0.39*x[1]^3*y[1]^2*y[3] + 0.42*x[2]^3*y[1]*y[3]^3 + 0.06*x[1]*x[2]^2*x[3]^3*y[1] + 0.14*x[1]^2*x[2]*x[3]*y[1]*y[2]*y[3] - 0.14*x[3]^2*y[1]^3*y[3]^3 + 0.93*x[3]^3*y[1]^3*y[3]^2 - 0.09*x[1]*x[2]*x[3]^2*y[2]^3*y[3]
Atomic measure: Atomic measure on the variables x[1], x[2], y[1], y[2] with 1 atoms:
 at [0.9999998974693745, 6.73355307206908e-8, 6.293070538555476e-8, 0.9999999015040413] with weight 0.9999997958983478
Status: OPTIMAL, Value: 1.29 [REDUCED] | reduce=0.001s, solver=0.1113s, total=0.731s
  SOS extraction succeeded; val = 1.29; total=0.731s
------------------------------------------------------------
Test 2 — seed=34 [REDUCED]
Polynom

50-element Vector{Any}:
 (1, 33, 1.2899998239739916, true, 0.0009999275207519531, 0.1113033, 0.7309999465942383, OPTIMAL)
 (2, 34, 0.5233202677028097, true, 0.0010001659393310547, 0.1242462, 0.7350001335144043, OPTIMAL)
 (3, 35, 1.3099999804161462, true, 0.0009999275207519531, 0.1025176, 0.7209999561309814, OPTIMAL)
 (4, 36, -0.6518519211376527, true, 0.0009999275207519531, 0.1253084, 0.7400000095367432, OPTIMAL)
 (5, 37, 0.48886626856674015, true, 0.0009999275207519531, 0.1376176, 0.748999834060669, OPTIMAL)
 (6, 38, 0.9099999992163994, true, 0.0, 0.1021474, 0.7219998836517334, OPTIMAL)
 (7, 39, 0.48000030513962105, true, 0.0, 0.2120452, 0.8310000896453857, OPTIMAL)
 (8, 40, 1.1399998370554185, true, 0.0, 0.1129474, 0.7280001640319824, OPTIMAL)
 (9, 41, 1.0399999540161906, true, 0.0, 0.0996388, 0.7149999141693115, OPTIMAL)
 (10, 42, 0.20999998239695167, true, 0.0, 0.1006964, 0.7159998416900635, OPTIMAL)
 ⋮
 (42, 74, 0.9141664882402714, true, 0.0, 0.124275, 0.744999885559082, OPTIMAL)


In [11]:
records = test_random_polynomials_reduced(1, varlists; degree=8, nterms=16, d_sel=8, tol=1e-1, seed=81)

------------------------------------------------------------
Test 1 — seed=81 [REDUCED]
Polynomial: 0.38 + 0.57*x[2]^2 - 0.52*x[1]*x[2]*y[1] - 0.72*x[2]^2*y[3]^2 + 0.92*x[1]*x[2]*y[1]^2 - 0.38*x[1]*x[2]*x[3]*y[2] + 0.43*x[1]*x[2]^2*y[1] + 0.01*x[1]*y[1]^2*y[2]*y[3] + 0.73*x[1]*x[2]^2*x[3]^2 + 0.07*x[3]*y[1]^2*y[2]*y[3]^2 - 0.82*x[1]*x[3]^2*y[1]^2*y[3] - 0.32*x[1]*x[2]^2*y[2]^2*y[3]^2 + 0.17*x[1]*x[2]^3*y[1]*y[2]*y[3] - 0.3*x[1]*x[2]^3*x[3]*y[3]^2 - 0.33*x[1]^2*x[3]*y[2]^2*y[3]^2 + 0.41*x[2]^2*x[3]^4*y[2]^2 + 0.24*x[1]*x[3]^2*y[1]^2*y[2]*y[3]^2
Atomic measure: Atomic measure on the variables x[1], x[2], y[1], y[2] with 2 atoms:
 at [3.6588826614229205e-6, 0.9999963369185146, 0.9067137083531458, 0.09323553648429386] with weight 0.6796139645976669
 at [-2.5133523416690753e-6, 1.0000024973127442, 0.18232599351988632, 0.8175890623059705] with weight 0.3196611254162191
Status: OPTIMAL, Value: 0.95 [REDUCED] | reduce=0.001s, solver=0.2231s, total=0.817s
  SOS extraction succeeded; val = 0.95;

1-element Vector{Any}:
 (1, 81, 0.9500001599386332, true, 0.0010001659393310547, 0.2230537, 0.817000150680542, OPTIMAL)

In [18]:
records = test_random_polynomials_reduced(n_tests, varlists; degree=10, nterms=16, d_sel=10, tol=3e-1, seed)

------------------------------------------------------------
Test 1 — seed=33 [REDUCED]
Polynomial: 0.94 - 0.34*y[2]*y[3] - 0.58*x[1]*y[2]^2*y[3] - 0.85*x[2]*y[2]*y[3]^3 + 0.06*x[1]*x[2]^2*x[3]^2 - 0.78*x[3]^2*y[1]^2*y[3]^2 - 0.31*x[2]*x[3]*y[1]^2*y[3]^2 + 0.96*x[1]*x[3]*y[2]^3*y[3] + 0.19*x[1]*x[3]*y[1]^4 - 0.09*x[1]^3*y[1]*y[3]^2 + 0.18*x[2]^2*y[1]*y[2]^2*y[3]^2 - 0.82*x[2]^2*x[3]^2*y[2]*y[3]^2 + 0.35*x[2]^3*x[3]*y[1]*y[2]*y[3]^3 - 0.31*x[1]*x[3]^2*y[1]^3*y[2]*y[3]^2 + 0.93*x[1]^2*x[2]*x[3]^2*y[1]^3*y[3] - 0.14*x[1]^2*x[2]^3*y[1]*y[2]*y[3]^2 - 0.77*x[1]^2*x[3]^2*y[1]^3*y[2]*y[3]^2
Atomic measure: Atomic measure on the variables x[1], x[2], y[1], y[2] with 1 atoms:
 at [0.49999733020686393, 1.49492854383036e-8, 0.9999999883599884, 0.0] with weight 0.9999999973799009
Status: OPTIMAL, Value: 0.9875 [REDUCED] | reduce=0.001s, solver=0.5115s, total=6.994s
  SOS extraction succeeded; val = 0.9875; total=6.994s
------------------------------------------------------------
Test 2 — seed=34 [R

50-element Vector{Any}:
 (1, 33, 0.9874999956957348, true, 0.0009999275207519531, 0.5115401, 6.99399995803833, OPTIMAL)
 (2, 34, 1.0099999864741271, true, 0.0, 0.4963075, 6.582000017166138, OPTIMAL)
 (3, 35, 0.5999999932942153, true, 0.0, 0.4103224, 6.483999967575073, OPTIMAL)
 (4, 36, 0.9899999957490043, true, 0.0, 0.4121064, 6.5350000858306885, OPTIMAL)
 (5, 37, 0.29999981637308526, true, 0.0010001659393310547, 0.5086632, 6.581000089645386, OPTIMAL)
 (6, 38, 0.0199999817542613, true, 0.0, 0.4541746, 6.58299994468689, OPTIMAL)
 (7, 39, 0.8099999246316091, true, 0.0, 0.4031288, 6.499000072479248, OPTIMAL)
 (8, 40, 0.4299999971572922, true, 0.0, 0.4568607, 6.634000062942505, OPTIMAL)
 (9, 41, 1.0700000001946615, true, 0.0, 0.3102363, 6.415999889373779, OPTIMAL)
 (10, 42, 0.13000001590580093, true, 0.0, 0.543831, 6.683000087738037, OPTIMAL)
 ⋮
 (42, 74, 0.415512803279255, true, 0.0, 0.4890592, 6.733000040054321, OPTIMAL)
 (43, 75, 0.4399999731117503, true, 0.0, 0.3541398, 7.1540000438690

### Multi-action Multi-infoset Absentminded Driver

In [12]:
# Multi-action Multi-infoset Absentminded Driver
# ℓ infosets, infoset I_j encountered d_j times, with m_j actions each
# Action set A_j = {c_j, a_{j,1},...,a_{j,m_j-1}}: c_j=continue, rest terminal
# Strategy x_j = (x_{j,0},...,x_{j,m_j-1}) ∈ Δ^{m_j-1}, x_{j,0}=continue prob
#
# Payoffs: r_{j,k,b} = α + β τ(j,k) + b + ε_{j,k,b}
#   α ~ Unif[-1,1], β ~ Unif[m_max+1, m_max+2], ε ~ Unif[-0.25,0.25]
#   r_term = -1
#
# u(x_1,...,x_ℓ) = Σ_j (Π_{h<j} x_{h,0}^{d_h})(Σ_{k=1}^{d_j} x_{j,0}^{k-1} Σ_{b=1}^{m_j-1} r_{j,k,b} x_{j,b})
#                + r_term Π_h x_{h,0}^{d_h}

function build_multi_action_utility(varlists, d_vec, r_payoffs, r_term)
    ℓ = length(varlists)
    u = zero(varlists[1][1])
    
    for j in 1:ℓ
        xj = varlists[j]       # (x_{j,0}, x_{j,1}, ..., x_{j,m_j-1})
        x0 = xj[1]             # continuation prob
        mj = length(xj)        # m_j actions total
        
        # Prefix: Π_{h<j} x_{h,0}^{d_h}
        prefix = one(x0)
        for h in 1:j-1
            prefix *= varlists[h][1]^d_vec[h]
        end
        
        # Σ_{k=1}^{d_j} x_{j,0}^{k-1} Σ_{b=1}^{m_j-1} r_{j,k,b} x_{j,b}
        inner = zero(x0)
        for k in 1:d_vec[j]
            for b in 1:(mj-1)
                inner += r_payoffs[j][k][b] * x0^(k-1) * xj[b+1]
            end
        end
        
        u += prefix * inner
    end
    
    # Terminal: r_term Π_h x_{h,0}^{d_h}
    term_prod = one(varlists[1][1])
    for h in 1:ℓ
        term_prod *= varlists[h][1]^d_vec[h]
    end
    u += r_term * term_prod
    
    return u
end

function sample_multi_action_payoffs(d_vec, m_vec)
    ℓ = length(d_vec)
    m_max = maximum(m_vec) - 1
    α = rand() * 2 - 1                      # Unif[-1,1]
    β = rand() + (m_max + 1)                # Unif[m_max+1, m_max+2]
    r_term = -1.0
    cum_d = vcat([0], cumsum(d_vec[1:end-1]))
    
    # r_payoffs[j][k][b] for j=1..ℓ, k=1..d_j, b=1..m_j-1
    r_payoffs = Vector{Vector{Vector{Float64}}}(undef, ℓ)
    for j in 1:ℓ
        r_payoffs[j] = Vector{Vector{Float64}}(undef, d_vec[j])
        for k in 1:d_vec[j]
            τ_jk = k + cum_d[j]
            r_payoffs[j][k] = Float64[]
            for b in 1:(m_vec[j]-1)
                ε = rand() * 0.5 - 0.25
                push!(r_payoffs[j][k], α + β * τ_jk + b + ε)
            end
        end
    end
    return r_payoffs, r_term
end

# Build varlists from m_vec: create a flat variable array, then partition by infoset
function make_varlists(m_vec)
    total = sum(m_vec)
    @polyvar a[1:total]
    vl = []
    idx = 1
    for mj in m_vec
        push!(vl, a[idx:idx+mj-1])
        idx += mj
    end
    return vl
end

function test_multi_action_driver(d_vec, m_vec; n_instances=100, seed=33, tol=1e-3)
    ℓ = length(d_vec)
    D = sum(d_vec)
    varlists = make_varlists(m_vec)

    Random.seed!(seed)

    sos_vals = Float64[]
    sos_atoms_found = Bool[]
    all_reduce = Float64[]
    all_solver = Float64[]
    all_total = Float64[]

    for inst in 1:n_instances
        r_payoffs, r_term = sample_multi_action_payoffs(d_vec, m_vec)
        u = build_multi_action_utility(varlists, d_vec, r_payoffs, r_term)
        
        nu, tval, status, t_reduce, t_solver, t_total = solve_polynomial_sos_reduced(u, varlists; d_sel=D, tol=tol)
        push!(sos_vals, tval)
        push!(sos_atoms_found, nu !== nothing)
        push!(all_reduce, t_reduce)
        push!(all_solver, t_solver)
        push!(all_total, t_total)
    end

    println("\n" * "="^70)
    println("Multi-action Driver: ℓ=$ℓ, d=$d_vec, m=$m_vec, D=$D ($n_instances instances) [REDUCED]")
    println("="^70)
    println("Atoms extracted: $(sum(sos_atoms_found)) / $n_instances")
    println("Reduce time: total=$(round(sum(all_reduce), digits=2))s, avg=$(round(mean(all_reduce), digits=4))s")
    println("Solver time: total=$(round(sum(all_solver), digits=2))s, avg=$(round(mean(all_solver), digits=4))s")
    println("Total time:  total=$(round(sum(all_total), digits=2))s, avg=$(round(mean(all_total), digits=4))s")

    return (sos_vals=sos_vals, sos_atoms_found=sos_atoms_found,
            all_reduce=all_reduce, all_solver=all_solver, all_total=all_total)
end

test_multi_action_driver (generic function with 1 method)

In [22]:
# ℓ=3, d=[2,2,2], m=[3,3,3], D=6
test_multi_action_driver([2, 2, 2], [2, 3, 4]; n_instances=100, seed=33)

Atomic measure: Atomic measure on the variables a[1], a[3], a[4], a[6], a[7], a[8] with 1 atoms:
 at [1.000000000102737, 0.9999999987609381, 0.0, 0.07413147364986551, 0.0, 1.0268512345706438e-7] with weight 1.0000000014985753
Status: OPTIMAL, Value: 27.932943 [REDUCED] | reduce=0.001s, solver=0.2393s, total=1.567s
Atomic measure: Atomic measure on the variables a[1], a[3], a[4], a[6], a[7], a[8] with 1 atoms:
 at [0.9999999996895514, 0.9999999947921928, 0.0, 0.07003554929049678, 2.3758673652061004e-8, 4.388882428310107e-8] with weight 1.0000000024739024
Status: OPTIMAL, Value: 27.400634 [REDUCED] | reduce=0.0s, solver=0.2116s, total=1.563s
Atomic measure: Atomic measure on the variables a[1], a[3], a[4], a[6], a[7], a[8] with 1 atoms:
 at [0.9999999999650236, 0.9999999996019109, 0.0, 0.06733450636031903, 0.0, 0.0] with weight 1.0000000057988547
Status: OPTIMAL, Value: 28.007055 [REDUCED] | reduce=0.0s, solver=0.2342s, total=1.562s
Atomic measure: Atomic measure on the variables a[1], a

(sos_vals = [27.93294259721017, 27.400634462798134, 28.00705549504971, 25.37230894772979, 27.56328481960639, 26.035884826512714, 27.480571841739543, 23.961666906949194, 23.462003983368756, 27.00876903631554  …  23.45239926634018, 26.067911019946116, 25.942895845812558, 24.04739260768103, 23.989684459718866, 27.810689418722696, 26.124289496860385, 24.161858086057897, 23.375477591077257, 26.6749307870519], sos_atoms_found = Bool[1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  1, 1, 1, 1, 1, 1, 1, 1, 1, 1], all_reduce = [0.0009999275207519531, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0009999275207519531, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0009999275207519531, 0.0, 0.0, 0.0], all_solver = [0.2392708, 0.2115928, 0.2341661, 0.2116729, 0.2278406, 0.2107699, 0.2303118, 0.2103425, 0.2089515, 0.2063122  …  0.2023082, 0.2070678, 0.2097783, 0.2026638, 0.2082503, 0.2321797, 0.2076228, 0.21059, 0.2098488, 0.2091799], all_total = [1.566999912261963, 1.562999963760376, 1.562000036239624, 1.551999807357788, 1.546000

## PGD Method

In [14]:
# PGD utilities for random polynomials on simplex
function proj_simplex(v::AbstractVector{<:Real})
    v = collect(float.(v))
    n = length(v)
    u = sort(v, rev=true)
    cssv = cumsum(u)
    rho = findlast(i -> u[i] + (1 - cssv[i]) / i > 0, 1:n)
    if rho === nothing
        return fill(1.0/n, n)
    end
    theta = (cssv[rho] - 1) / rho
    return max.(v .- theta, 0.0)
end

function project_onto_feasible(vars::AbstractVector, varlists)
    out = similar(vars, Float64)
    idx = 1
    for group in varlists
        n = length(group)
        out[idx:idx+n-1] .= proj_simplex(vars[idx:idx+n-1])
        idx += n
    end
    return out
end

function poly_value(p, allvars, vals)
    return convert(Float64, subs(p, allvars => vals))
end

function poly_grad_polys(p, allvars)
    return [differentiate(p, v) for v in allvars]
end

function poly_grad_value(grad_polys, allvars, vals)
    return [convert(Float64, subs(g, allvars => vals)) for g in grad_polys]
end

function pgd_maximize_polynomial(p, varlists;
    learning_rate=0.02, max_iterations=5000, tolerance=1e-8, seed=1)
    allvars = reduce(vcat, varlists)
    grad_polys = poly_grad_polys(p, allvars)

    Random.seed!(seed)
    parts = Vector{Float64}()
    for group in varlists
        v = rand(length(group))
        v ./= sum(v)
        append!(parts, v)
    end
    vars = project_onto_feasible(parts, varlists)

    history = Vector{Vector{Float64}}()
    obj_history = Float64[]
    push!(history, copy(vars))
    push!(obj_history, -poly_value(p, allvars, vars))

    start_time = time()
    for iter in 1:max_iterations
        g = poly_grad_value(grad_polys, allvars, vars)
        vars_new = vars + learning_rate .* g  # ascent on p
        vars_new = project_onto_feasible(vars_new, varlists)
        push!(history, copy(vars_new))
        push!(obj_history, -poly_value(p, allvars, vars_new))
        if norm(vars_new - vars) < tolerance
            vars = vars_new
            break
        end
        vars = vars_new
    end
    elapsed = time() - start_time
    final_val = poly_value(p, allvars, vars)
    return vars, final_val, history, obj_history, elapsed
end

println("PGD utilities ready.")

# Test PGD performance on randomly generated polynomials
function test_pgd_polynomials(n_tests, varlists; degree, nterms, pgd_runs=100, seed=33)
    pgd_records = []
    all_avg_times = []
    all_rates = []
    for i in 1:n_tests
        s = seed + i - 1
        p = random_sos_poly(varlists; degree=degree, nterms=nterms, seed=s)
        results = []
        point_groups = Dict{Tuple{Vector{Float64}, Vector{Float64}}, NamedTuple{(:cnt, :sum_val, :sum_elapsed)}}()
        for r in 1:pgd_runs
            vars, val, history, obj_history, elapsed = pgd_maximize_polynomial(p, varlists;
                learning_rate=0.02, max_iterations=5000, tolerance=1e-8, seed=s*1000 + r)
            push!(results, (val=val, vars=vars, elapsed=elapsed))
            x_rounded = round.(vars[1:3], digits=4)
            y_rounded = round.(vars[4:6], digits=4)
            key = (x_rounded, y_rounded)
            if haskey(point_groups, key)
                point_groups[key] = (cnt = point_groups[key].cnt + 1, 
                                     sum_val = point_groups[key].sum_val + val, 
                                     sum_elapsed = point_groups[key].sum_elapsed + elapsed)
            else
                point_groups[key] = (cnt = 1, sum_val = val, sum_elapsed = elapsed)
            end
        end
        
        max_val = maximum(r.val for r in results)
        total_time = sum(r.elapsed for r in results)
        avg_time = total_time / pgd_runs
        optimum_count = count(r.val >= max_val - 0.001 for r in results)
        rate = optimum_count / pgd_runs
        
        push!(all_avg_times, avg_time)
        push!(all_rates, rate)
        
        println("PGD test $i: total time=$(round(total_time, digits=2))s, avg time=$(round(avg_time, digits=4))s, optimum rate=$(round(rate*100, digits=2))%")
        println("Optimal convergence point (val >= $(round(max_val - 0.001, digits=6))):")
        optimal_groups = filter(x -> x[2].sum_val / x[2].cnt >= max_val - 0.001, point_groups)
        sorted_optimal = sort(collect(optimal_groups), by = x -> -x[2].sum_val / x[2].cnt)
        if !isempty(sorted_optimal)
            key, stats = sorted_optimal[1]
            x_avg = key[1]
            y_avg = key[2]
            avg_val = round(stats.sum_val / stats.cnt, digits=6)
            cnt = stats.cnt
            prop = cnt / pgd_runs
            println("  x: $(x_avg), y: $(y_avg), val: $(avg_val), $(cnt)/100 ($(round(prop*100, digits=2))%)")
        end
        push!(pgd_records, (i=i, seed=s, results=results, point_groups=point_groups, max_val=max_val, total_time=total_time, avg_time=avg_time, rate=rate))
    end

    overall_avg_time = mean(all_avg_times)
    overall_rate = mean(all_rates)
    println("\nOverall: avg time per run=$(round(overall_avg_time, digits=4))s, avg optimum rate=$(round(overall_rate*100, digits=2))%")
    return pgd_records
end

PGD utilities ready.


test_pgd_polynomials (generic function with 1 method)

In [15]:
# Test PGD performance on randomly generated polynomials
function test_pgd_polynomials(n_tests, varlists; degree, nterms, pgd_runs=100, seed=33)
    pgd_records = []
    all_avg_times = []
    all_rates = []
    for i in 1:n_tests
        s = seed + i - 1
        p = random_sos_poly(varlists; degree=degree, nterms=nterms, seed=s)
        results = []
        point_groups = Dict{Tuple{Vector{Float64}, Vector{Float64}}, NamedTuple{(:cnt, :sum_val, :sum_elapsed)}}()
        for r in 1:pgd_runs
            vars, val, history, obj_history, elapsed = pgd_maximize_polynomial(p, varlists;
                learning_rate=0.02, max_iterations=5000, tolerance=1e-8, seed=s*1000 + r)
            push!(results, (val=val, vars=vars, elapsed=elapsed))
            x_rounded = round.(vars[1:3], digits=4)
            y_rounded = round.(vars[4:6], digits=4)
            key = (x_rounded, y_rounded)
            if haskey(point_groups, key)
                point_groups[key] = (cnt = point_groups[key].cnt + 1, 
                                     sum_val = point_groups[key].sum_val + val, 
                                     sum_elapsed = point_groups[key].sum_elapsed + elapsed)
            else
                point_groups[key] = (cnt = 1, sum_val = val, sum_elapsed = elapsed)
            end
        end
        
        max_val = maximum(r.val for r in results)
        total_time = sum(r.elapsed for r in results)
        avg_time = total_time / pgd_runs
        optimum_count = count(r.val >= max_val - 0.001 for r in results)
        rate = optimum_count / pgd_runs
        
        push!(all_avg_times, avg_time)
        push!(all_rates, rate)
        
        println("PGD test $i: total time=$(round(total_time, digits=2))s, avg time=$(round(avg_time, digits=4))s, optimum rate=$(round(rate*100, digits=2))%")
        println("Optimal convergence point (val >= $(round(max_val - 0.001, digits=6))):")
        optimal_groups = filter(x -> x[2].sum_val / x[2].cnt >= max_val - 0.001, point_groups)
        sorted_optimal = sort(collect(optimal_groups), by = x -> -x[2].sum_val / x[2].cnt)
        if !isempty(sorted_optimal)
            key, stats = sorted_optimal[1]
            x_avg = key[1]
            y_avg = key[2]
            avg_val = round(stats.sum_val / stats.cnt, digits=6)
            cnt = stats.cnt
            prop = cnt / pgd_runs
            println("  x: $(x_avg), y: $(y_avg), val: $(avg_val), $(cnt)/100 ($(round(prop*100, digits=2))%)")
        end
        push!(pgd_records, (i=i, seed=s, results=results, point_groups=point_groups, max_val=max_val, total_time=total_time, avg_time=avg_time, rate=rate))
    end

    overall_avg_time = mean(all_avg_times)
    overall_rate = mean(all_rates)
    println("\nOverall: avg time per run=$(round(overall_avg_time, digits=4))s, avg optimum rate=$(round(overall_rate*100, digits=2))%")
    return pgd_records
end

test_pgd_polynomials (generic function with 1 method)

In [37]:
pgd_records = test_pgd_polynomials(n_tests, varlists; degree=4, nterms=16)

PGD test 1: total time=3.23s, avg time=0.0323s, optimum rate=90.0%
Optimal convergence point (val >= -0.231):
  x: [0.0, 0.0, 1.0], y: [1.0, 0.0, 0.0], val: -0.23, 90/100 (90.0%)
PGD test 2: total time=3.59s, avg time=0.0359s, optimum rate=79.0%
Optimal convergence point (val >= 1.919):
  x: [0.0, 0.0, 1.0], y: [0.0, 1.0, 0.0], val: 1.92, 79/100 (79.0%)
PGD test 3: total time=0.98s, avg time=0.0098s, optimum rate=34.0%
Optimal convergence point (val >= 1.279):
  x: [1.0, 0.0, 0.0], y: [0.0, 1.0, 0.0], val: 1.28, 34/100 (34.0%)
PGD test 4: total time=1.53s, avg time=0.0153s, optimum rate=98.0%
Optimal convergence point (val >= 1.349):
  x: [1.0, 0.0, 0.0], y: [1.0, 0.0, 0.0], val: 1.35, 98/100 (98.0%)
PGD test 5: total time=1.16s, avg time=0.0116s, optimum rate=100.0%
Optimal convergence point (val >= 2.219):
  x: [0.0, 0.0, 1.0], y: [0.0, 0.0, 1.0], val: 2.22, 100/100 (100.0%)
PGD test 6: total time=11.41s, avg time=0.1141s, optimum rate=16.0%
Optimal convergence point (val >= 1.655729

50-element Vector{Any}:
 (i = 1, seed = 33, results = Any[(val = -0.22999999999999998, vars = [0.0, 0.0, 1.0, 1.0, 0.0, 0.0], elapsed = 0.020999908447265625), (val = -0.22999999999999998, vars = [0.0, 0.0, 1.0, 1.0, 0.0, 0.0], elapsed = 0.08800005912780762), (val = -0.7232833044151731, vars = [0.0, 1.0, 0.0, 0.0, 0.32200160719366605, 0.677998392806334], elapsed = 0.15199995040893555), (val = -0.7232833044151696, vars = [0.0, 1.0, 0.0, 0.0, 0.3220016025817132, 0.6779983974182869], elapsed = 0.18000006675720215), (val = -0.22999999999999998, vars = [0.0, 0.0, 1.0, 1.0, 0.0, 0.0], elapsed = 0.015000104904174805), (val = -0.22999999999999998, vars = [0.0, 0.0, 1.0, 1.0, 0.0, 0.0], elapsed = 0.010999917984008789), (val = -0.22999999999999998, vars = [0.0, 0.0, 1.0, 1.0, 0.0, 0.0], elapsed = 0.013000011444091797), (val = -0.22999999999999998, vars = [0.0, 0.0, 1.0, 1.0, 0.0, 0.0], elapsed = 0.01100015640258789), (val = -0.22999999999999998, vars = [0.0, 0.0, 1.0, 1.0, 0.0, 0.0], elapsed = 0.

In [39]:
pgd_records = test_pgd_polynomials(n_tests, varlists; degree=6, nterms=16)

PGD test 1: total time=5.56s, avg time=0.0556s, optimum rate=100.0%
Optimal convergence point (val >= 0.916382):
  x: [1.0, 0.0, 0.0], y: [0.7553, 0.0, 0.2447], val: 0.917382, 100/100 (100.0%)
PGD test 2: total time=4.41s, avg time=0.0441s, optimum rate=100.0%
Optimal convergence point (val >= 0.569):
  x: [0.0, 1.0, 0.0], y: [1.0, 0.0, 0.0], val: 0.57, 100/100 (100.0%)
PGD test 3: total time=15.46s, avg time=0.1546s, optimum rate=85.0%
Optimal convergence point (val >= -0.0135):
  x: [1.0, 0.0, 0.0], y: [0.5, 0.0, 0.5], val: -0.0125, 85/100 (85.0%)
PGD test 4: total time=22.38s, avg time=0.2238s, optimum rate=100.0%
Optimal convergence point (val >= 1.539):
  x: [0.0, 0.0, 1.0], y: [0.0, 0.0, 1.0], val: 1.54, 100/100 (100.0%)
PGD test 5: total time=6.97s, avg time=0.0697s, optimum rate=51.0%
Optimal convergence point (val >= 0.233138):
  x: [0.2759, 0.7241, 0.0], y: [1.0, 0.0, 0.0], val: 0.234138, 51/100 (51.0%)
PGD test 6: total time=12.16s, avg time=0.1216s, optimum rate=100.0%
Opti

50-element Vector{Any}:
 (i = 1, seed = 33, results = Any[(val = 0.917381926926848, vars = [1.0, 0.0, 0.0, 0.7553002357685249, 0.0, 0.24469976423147505], elapsed = 0.05299997329711914), (val = 0.9173819269268512, vars = [1.0, 0.0, 0.0, 0.7553002404045299, 0.0, 0.24469975959546997], elapsed = 0.04900002479553223), (val = 0.9173819269268504, vars = [1.0, 0.0, 0.0, 0.7553002392686775, 0.0, 0.24469976073132244], elapsed = 0.10100007057189941), (val = 0.9173819269268488, vars = [1.0, 0.0, 0.0, 0.7553002369673504, 0.0, 0.24469976303264968], elapsed = 0.04999995231628418), (val = 0.9173819269268482, vars = [1.0, 0.0, 0.0, 0.7553002361813452, 0.0, 0.24469976381865466], elapsed = 0.04800009727478027), (val = 0.917381926926852, vars = [1.0, 0.0, 0.0, 0.7553002419466939, 0.0, 0.24469975805330602], elapsed = 0.05200004577636719), (val = 0.9173819269268494, vars = [1.0, 0.0, 0.0, 0.755300237817708, 0.0, 0.24469976218229184], elapsed = 0.048999786376953125), (val = 0.9173819269268496, vars = [1.0, 0

In [41]:
pgd_records = test_pgd_polynomials(n_tests, varlists; degree=8, nterms=16)

PGD test 1: total time=3.9s, avg time=0.039s, optimum rate=99.0%
Optimal convergence point (val >= 1.289):
  x: [1.0, 0.0, 0.0], y: [0.0, 1.0, 0.0], val: 1.29, 99/100 (99.0%)
PGD test 2: total time=41.62s, avg time=0.4162s, optimum rate=8.0%
Optimal convergence point (val >= 0.52232):
  x: [0.25, 0.75, 0.0], y: [0.0, 1.0, 0.0], val: 0.52332, 8/100 (8.0%)
PGD test 3: total time=4.7s, avg time=0.047s, optimum rate=100.0%
Optimal convergence point (val >= 1.309):
  x: [0.0, 0.0, 1.0], y: [1.0, 0.0, 0.0], val: 1.31, 100/100 (100.0%)
PGD test 4: total time=21.48s, avg time=0.2148s, optimum rate=60.0%
Optimal convergence point (val >= -0.652852):
  x: [1.0, 0.0, 0.0], y: [0.0, 0.6667, 0.3333], val: -0.651852, 60/100 (60.0%)
PGD test 5: total time=9.11s, avg time=0.0911s, optimum rate=98.0%
Optimal convergence point (val >= 0.487866):
  x: [0.7384, 0.0, 0.2616], y: [1.0, 0.0, 0.0], val: 0.488866, 98/100 (98.0%)
PGD test 6: total time=2.18s, avg time=0.0218s, optimum rate=100.0%
Optimal conver

50-element Vector{Any}:
 (i = 1, seed = 33, results = Any[(val = 1.29, vars = [1.0, 0.0, 0.0, 0.0, 1.0, 0.0], elapsed = 0.04400014877319336), (val = 1.29, vars = [1.0, 0.0, 0.0, 0.0, 1.0, 0.0], elapsed = 0.07899999618530273), (val = 1.29, vars = [1.0, 0.0, 0.0, 0.0, 1.0, 0.0], elapsed = 0.018999814987182617), (val = 1.29, vars = [1.0, 0.0, 0.0, 0.0, 1.0, 0.0], elapsed = 0.10600018501281738), (val = 1.29, vars = [1.0, 0.0, 0.0, 0.0, 1.0, 0.0], elapsed = 0.0279998779296875), (val = 1.29, vars = [1.0, 0.0, 0.0, 0.0, 1.0, 0.0], elapsed = 0.023000001907348633), (val = 1.29, vars = [1.0, 0.0, 0.0, 0.0, 1.0, 0.0], elapsed = 0.03500008583068848), (val = 1.29, vars = [1.0, 0.0, 0.0, 0.0, 1.0, 0.0], elapsed = 0.017999887466430664), (val = 1.29, vars = [1.0, 0.0, 0.0, 0.0, 1.0, 0.0], elapsed = 0.02500009536743164), (val = 1.29, vars = [1.0, 0.0, 0.0, 0.0, 1.0, 0.0], elapsed = 0.03500008583068848)  …  (val = 1.29, vars = [1.0, 0.0, 0.0, 0.0, 1.0, 0.0], elapsed = 0.023000001907348633), (val = 1.29,

In [30]:
pgd_records = test_pgd_polynomials(n_tests, varlists; degree=10, nterms=16)

PGD test 1: total time=53.84s, avg time=0.5384s, optimum rate=81.0%
Optimal convergence point (val >= 0.9865):
  x: [0.5, 0.0, 0.5], y: [1.0, 0.0, 0.0], val: 0.9875, 76/100 (76.0%)
PGD test 2: total time=35.48s, avg time=0.3548s, optimum rate=98.0%
Optimal convergence point (val >= 1.009):
  x: [0.5, 0.0, 0.5], y: [0.0, 1.0, 0.0], val: 1.01, 98/100 (98.0%)
PGD test 3: total time=3.05s, avg time=0.0305s, optimum rate=94.0%
Optimal convergence point (val >= 0.599):
  x: [0.0, 0.0, 1.0], y: [1.0, 0.0, 0.0], val: 0.6, 94/100 (94.0%)
PGD test 4: total time=1.78s, avg time=0.0178s, optimum rate=100.0%
Optimal convergence point (val >= 0.989):
  x: [0.0, 0.0, 1.0], y: [1.0, 0.0, 0.0], val: 0.99, 100/100 (100.0%)
PGD test 5: total time=3.87s, avg time=0.0387s, optimum rate=93.0%
Optimal convergence point (val >= 0.299):
  x: [1.0, 0.0, 0.0], y: [0.0, 1.0, 0.0], val: 0.3, 93/100 (93.0%)
PGD test 6: total time=1.68s, avg time=0.0168s, optimum rate=100.0%
Optimal convergence point (val >= 0.019):

50-element Vector{Any}:
 (i = 1, seed = 33, results = Any[(val = 0.9401810670806694, vars = [0.37539001653816423, 0.2235718204297639, 0.4010381630320719, 0.0, 0.0, 1.0], elapsed = 0.6740000247955322), (val = 0.9874999999993495, vars = [0.4999981496612186, 0.0, 0.5000018503387813, 1.0, 0.0, 0.0], elapsed = 0.4609999656677246), (val = 0.9874999999993478, vars = [0.5000018527277539, 0.0, 0.4999981472722461, 1.0, 0.0, 0.0], elapsed = 0.44700002670288086), (val = 0.9874999999993495, vars = [0.4999981496568169, 0.0, 0.5000018503431831, 1.0, 0.0, 0.0], elapsed = 0.5590000152587891), (val = 0.9874999999993488, vars = [0.5000018512085322, 0.0, 0.49999814879146776, 1.0, 0.0, 0.0], elapsed = 0.5399999618530273), (val = 0.9404190896485524, vars = [0.23440010056215446, 0.2853215349766972, 0.4802783644611483, 0.29176041049645013, 0.7082395895035498, 0.0], elapsed = 0.6579999923706055), (val = 0.9402211720290905, vars = [0.34708419850702, 0.26713630166389346, 0.3857794998290867, 0.0, 0.0, 1.0], elaps

### PGD more variables

In [17]:
# Generalized PGD test function for multiple variable groups
function test_pgd_polynomials_general(n_tests, varlists; degree, nterms, pgd_runs=100, seed=33)
    pgd_records = []
    all_avg_times = []
    all_rates = []
    for i in 1:n_tests
        s = seed + i - 1
        p = random_sos_poly(varlists; degree=degree, nterms=nterms, seed=s)
        results = []
        # Compute group lengths for slicing
        group_lengths = [length(group) for group in varlists]
        cum_lengths = cumsum(group_lengths)
        # Use a dict with tuple of tuples as key
        point_groups = Dict{NTuple{length(varlists), Vector{Float64}}, NamedTuple{(:cnt, :sum_val, :sum_elapsed)}}()
        for r in 1:pgd_runs
            vars, val, history, obj_history, elapsed = pgd_maximize_polynomial(p, varlists;
                learning_rate=0.02, max_iterations=5000, tolerance=1e-8, seed=s*1000 + r)
            push!(results, (val=val, vars=vars, elapsed=elapsed))
            # Round each group
            rounded_groups = []
            start_idx = 1
            for len in group_lengths
                push!(rounded_groups, round.(vars[start_idx:start_idx+len-1], digits=4))
                start_idx += len
            end
            key = tuple(rounded_groups...)
            if haskey(point_groups, key)
                point_groups[key] = (cnt = point_groups[key].cnt + 1, 
                                     sum_val = point_groups[key].sum_val + val, 
                                     sum_elapsed = point_groups[key].sum_elapsed + elapsed)
            else
                point_groups[key] = (cnt = 1, sum_val = val, sum_elapsed = elapsed)
            end
        end
        
        max_val = maximum(r.val for r in results)
        total_time = sum(r.elapsed for r in results)
        avg_time = total_time / pgd_runs
        optimum_count = count(r.val >= max_val - 0.001 for r in results)
        rate = optimum_count / pgd_runs
        
        push!(all_avg_times, avg_time)
        push!(all_rates, rate)
        
        println("PGD test $i: total time=$(round(total_time, digits=2))s, avg time=$(round(avg_time, digits=4))s, optimum rate=$(round(rate*100, digits=2))%")
        println("Optimal convergence point (val >= $(round(max_val - 0.001, digits=6))):")
        optimal_groups = filter(x -> x[2].sum_val / x[2].cnt >= max_val - 0.001, point_groups)
        sorted_optimal = sort(collect(optimal_groups), by = x -> -x[2].sum_val / x[2].cnt)
        if !isempty(sorted_optimal)
            key, stats = sorted_optimal[1]
            avg_val = round(stats.sum_val / stats.cnt, digits=6)
            cnt = stats.cnt
            prop = cnt / pgd_runs
            println("  Groups: $(key), val: $(avg_val), $(cnt)/$pgd_runs ($(round(prop*100, digits=2))%)")
        end
        push!(pgd_records, (i=i, seed=s, results=results, point_groups=point_groups, max_val=max_val, total_time=total_time, avg_time=avg_time, rate=rate))
    end

    overall_avg_time = mean(all_avg_times)
    overall_rate = mean(all_rates)
    println("\nOverall: avg time per run=$(round(overall_avg_time, digits=4))s, avg optimum rate=$(round(overall_rate*100, digits=2))%")
    return pgd_records
end

test_pgd_polynomials_general (generic function with 1 method)

In [ ]:
pgd_records_general = test_pgd_polynomials_general(n_tests, varlists_var3; degree=4, nterms=16)

PGD test 1: total time=8.9s, avg time=0.089s, optimum rate=4.0%
Optimal convergence point (val >= 0.087396):
  Groups: ([0.0, 1.0, 0.0], [0.0, 1.0, 0.0], [0.8881, 0.0, 0.1119]), val: 0.088396, 4/100 (4.0%)
PGD test 2: total time=1.65s, avg time=0.0165s, optimum rate=100.0%
Optimal convergence point (val >= 2.939):
  Groups: ([0.0, 0.0, 1.0], [0.0, 1.0, 0.0], [1.0, 0.0, 0.0]), val: 2.94, 100/100 (100.0%)
PGD test 3: total time=3.39s, avg time=0.0339s, optimum rate=85.0%
Optimal convergence point (val >= 0.729):
  Groups: ([0.0, 1.0, 0.0], [0.0, 0.0, 1.0], [0.0, 1.0, 0.0]), val: 0.73, 85/100 (85.0%)
PGD test 4: total time=2.7s, avg time=0.027s, optimum rate=89.0%
Optimal convergence point (val >= 1.259):
  Groups: ([0.0, 0.0, 1.0], [0.0, 0.0, 1.0], [0.0, 1.0, 0.0]), val: 1.26, 89/100 (89.0%)
PGD test 5: total time=2.78s, avg time=0.0278s, optimum rate=35.0%
Optimal convergence point (val >= 0.009):
  Groups: ([0.0, 0.0, 1.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]), val: 0.01, 35/100 (35.0%)
P

50-element Vector{Any}:
 (i = 1, seed = 33, results = Any[(val = -0.5300000000000001, vars = [1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0], elapsed = 0.15899991989135742), (val = -0.17000000000000004, vars = [0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0], elapsed = 0.08800005912780762), (val = -0.25, vars = [1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.4605588606234113, 0.5394411393765887, 0.0], elapsed = 0.01399993896484375), (val = 0.07999999999999985, vars = [0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.5959208299677801, 0.4040791700322199, 0.0], elapsed = 0.07500004768371582), (val = 0.07999999999999985, vars = [0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.05090937155134906, 0.949090628448651, 0.0], elapsed = 0.024000167846679688), (val = -0.17000000000000004, vars = [0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0], elapsed = 0.03399991989135742), (val = -0.17000000000000004, vars = [0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0], elapsed = 0.08800005912780762), (val = -0.17000000000000004, vars = [0.0, 0.0, 1.0, 0.0, 1.0, 0.

In [27]:
pgd_records_general = test_pgd_polynomials_general(n_tests, varlists_var4; degree=4, nterms=16)

PGD test 1: total time=17.42s, avg time=0.1742s, optimum rate=5.0%
Optimal convergence point (val >= 1.929):
  Groups: ([0.0, 1.0, 0.0], [0.0, 0.0, 1.0], [0.0, 1.0, 0.0], [0.0, 1.0, 0.0]), val: 1.93, 5/100 (5.0%)
PGD test 2: total time=4.31s, avg time=0.0431s, optimum rate=87.0%
Optimal convergence point (val >= 1.609):
  Groups: ([0.0, 1.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0], [0.0, 1.0, 0.0]), val: 1.61, 87/100 (87.0%)
PGD test 3: total time=3.54s, avg time=0.0354s, optimum rate=82.0%
Optimal convergence point (val >= 2.679):
  Groups: ([1.0, 0.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 1.0], [0.0, 1.0, 0.0]), val: 2.68, 82/100 (82.0%)
PGD test 4: total time=12.74s, avg time=0.1274s, optimum rate=83.0%
Optimal convergence point (val >= 1.829):
  Groups: ([0.0, 0.3969, 0.6031], [0.0, 1.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]), val: 1.83, 1/100 (1.0%)
PGD test 5: total time=3.88s, avg time=0.0388s, optimum rate=9.0%
Optimal convergence point (val >= 0.289):
  Groups: ([0.0, 1.0, 0.0], [

50-element Vector{Any}:
 (i = 1, seed = 33, results = Any[(val = 1.9300000000000002, vars = [0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0], elapsed = 0.11800003051757812), (val = 1.8602499999997866, vars = [0.0, 1.0, 0.0, 0.3774800100474784, 0.6225192809049755, 7.090475460843292e-7, 0.0, 1.0, 0.0, 0.18333333425986675, 0.0, 0.8166666657401334], elapsed = 0.18899989128112793), (val = 1.860249999999788, vars = [0.0, 1.0, 0.0, 0.47369722189207847, 0.5263020722501862, 7.058577353045833e-7, 0.0, 1.0, 0.0, 0.18333335728122657, 0.0, 0.8166666427187734], elapsed = 0.17499995231628418), (val = 1.8602499999997881, vars = [0.0, 1.0, 0.0, 0.4276851966746617, 0.5723140968888699, 7.064364683829714e-7, 0.0, 1.0, 0.0, 0.18333333365637144, 0.0, 0.8166666663436285], elapsed = 0.1809999942779541), (val = 1.8602499999997864, vars = [0.0, 1.0, 0.0, 0.4231386015994681, 0.5768606889961786, 7.094043533282062e-7, 0.0, 1.0, 0.0, 0.1833333343211132, 0.0, 0.8166666656788868], elapsed = 0.185000181198

In [28]:
pgd_records_general = test_pgd_polynomials_general(n_tests, varlists_var5; degree=4, nterms=16)

PGD test 1: total time=18.02s, avg time=0.1802s, optimum rate=58.0%
Optimal convergence point (val >= 2.319):
  Groups: ([1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0], [0.0, 1.0, 0.0], [0.0, 1.0, 0.0]), val: 2.32, 58/100 (58.0%)
PGD test 2: total time=8.29s, avg time=0.0829s, optimum rate=98.0%
Optimal convergence point (val >= 2.349):
  Groups: ([0.0, 0.0, 1.0], [0.3193, 0.2267, 0.454], [1.0, 0.0, 0.0], [0.0, 0.0, 1.0], [0.0, 1.0, 0.0]), val: 2.35, 1/100 (1.0%)
PGD test 3: total time=17.32s, avg time=0.1732s, optimum rate=88.0%
Optimal convergence point (val >= 1.373533):
  Groups: ([0.4121, 0.5879, 0.0], [0.8033, 0.0, 0.1967], [1.0, 0.0, 0.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0]), val: 1.374533, 1/100 (1.0%)
PGD test 4: total time=8.59s, avg time=0.0859s, optimum rate=100.0%
Optimal convergence point (val >= 2.939):
  Groups: ([1.0, 0.0, 0.0], [1.0, 0.0, 0.0], [1.0, 0.0, 0.0], [1.0, 0.0, 0.0], [1.0, 0.0, 0.0]), val: 2.94, 100/100 (100.0%)
PGD test 5: total time=14.26s, avg time=0.14

50-element Vector{Any}:
 (i = 1, seed = 33, results = Any[(val = 2.1, vars = [1.0, 0.0, 0.0, 3.700743415417188e-17, 0.2716863051618806, 0.7283136948381194, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0], elapsed = 0.03399991989135742), (val = 2.32, vars = [1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0], elapsed = 0.07800006866455078), (val = 2.1, vars = [1.0, 0.0, 0.0, 0.147630002059087, 0.6434488897619691, 0.2089211081789439, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0], elapsed = 0.1679999828338623), (val = 2.32, vars = [1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0], elapsed = 0.40000009536743164), (val = 2.1, vars = [1.0, 0.0, 0.0, 0.2722901891796114, 0.45722721236667024, 0.2704825984537184, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0], elapsed = 0.04900002479553223), (val = 2.32, vars = [1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0], elapsed = 0.33299994468688965), (val = 2.32, vars = [1.0, 0.0, 0.0

In [ ]:
pgd_records_general = test_pgd_polynomials_general(n_tests, varlists_var6; degree=4, nterms=16)

PGD test 1: total time=32.51s, avg time=0.3251s, optimum rate=59.0%
Optimal convergence point (val >= 2.659):
  Groups: ([0.0, 1.0, 0.0], [1.0, 0.0, 0.0], [0.4505, 0.0, 0.5495], [0.0, 1.0, 0.0], [0.0, 0.4439, 0.5561], [0.0, 1.0, 0.0]), val: 2.66, 1/100 (1.0%)
PGD test 2: total time=24.71s, avg time=0.2471s, optimum rate=93.0%
Optimal convergence point (val >= 1.549):
  Groups: ([0.0, 1.0, 0.0], [0.5, 0.5, 0.0], [1.0, 0.0, 0.0], [1.0, 0.0, 0.0], [1.0, 0.0, 0.0], [0.7962, 0.1519, 0.0519]), val: 1.55, 1/100 (1.0%)
PGD test 3: total time=27.02s, avg time=0.2702s, optimum rate=46.0%
Optimal convergence point (val >= 2.549):
  Groups: ([0.0, 1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0], [1.0, 0.0, 0.0]), val: 2.55, 46/100 (46.0%)
PGD test 4: total time=8.15s, avg time=0.0815s, optimum rate=43.0%
Optimal convergence point (val >= 3.289):
  Groups: ([0.0, 0.0, 1.0], [0.0, 1.0, 0.0], [0.0, 1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [1.0, 0.0, 0.0]), val: 3.29,

50-element Vector{Any}:
 (i = 1, seed = 33, results = Any[(val = 2.61, vars = [0.0, 1.0, 0.0, 0.0, 0.07423460883216766, 0.9257653911678323, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.6486169199697026, 0.12378699353519307, 0.22759608649510438, 0.0, 1.0, 0.0], elapsed = 0.18899989128112793), (val = 2.61, vars = [0.0, 1.0, 0.0, 0.0, 0.4387568142173673, 0.5612431857826328, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.4347282414928487, 0.28191794557212413, 0.2833538129350272, 0.0, 1.0, 0.0], elapsed = 0.46700000762939453), (val = 2.66, vars = [0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.2488693065867134, 0.0, 0.7511306934132866, 0.0, 1.0, 0.0, 0.0, 0.304431172684466, 0.695568827315534, 0.0, 1.0, 0.0], elapsed = 0.17300009727478027), (val = 2.66, vars = [0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.5342872806272789, 0.465712719372721, 0.0, 1.0, 0.0], elapsed = 0.28600001335144043), (val = 2.61, vars = [0.0, 1.0, 0.0, 0.0, 0.5036989838709786, 0.4963010161290215, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.487991994058